# Resume Screen Crew

In [20]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [21]:
from crewai import Agent, Task, Crew

In [38]:
import os

openai_api_key = os.getenv("OPENAI_API_KEY")
# print(openai_api_key)
serper_api_key = os.getenv("SERPER_API_KEY")
# print(serper_api_key)
# openai_model_name = os.getenv("OPENAI_MODEL_NAME")
# print(openai_model_name)
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

## CrewAI Tools

In [23]:
from crewai_tools import (
  FileReadTool,
  ScrapeWebsiteTool,
  MDXSearchTool,
  SerperDevTool
)

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()
read_resume = FileReadTool(file_path='./fake_resume.md')
semantic_search_resume = MDXSearchTool(mdx='./fake_resume.md')

In [ ]:
from IPython.display import Markdown, display
display(Markdown("./fake_resume.md"))

## Creating Agents

In [25]:
# Agent 1: Researcher
researcher = Agent(
    role="Tech Job Researcher",
    goal="Make sure to do amazing analysis on "
         "job posting to help job applicants",
    tools = [scrape_tool, search_tool],
    verbose=True,
    backstory=(
        "As a Job Researcher, your prowess in "
        "navigating and extracting critical "
        "information from job postings is unmatched."
        "Your skills help pinpoint the necessary "
        "qualifications and skills sought "
        "by employers, forming the foundation for "
        "effective application tailoring."
    )
)

In [26]:
# Agent 2: Profiler
profiler = Agent(
    role="Personal Profiler for Engineers",
    goal="Do increditble research on job applicants "
         "to help them stand out in the job market",
    tools = [scrape_tool, search_tool,
             read_resume, semantic_search_resume],
    verbose=True,
    backstory=(
        "Equipped with analytical prowess, you dissect "
        "and synthesize information "
        "from diverse sources to craft comprehensive "
        "personal and professional profiles, laying the "
        "groundwork for personalized resume enhancements."
    )
)

In [27]:
# Agent 3: Resume Strategist
resume_strategist = Agent(
    role="Resume Strategist for Engineers",
    goal="Find all the best ways to make a "
         "resume stand out in the job market.",
    tools = [scrape_tool, search_tool,
             read_resume, semantic_search_resume],
    verbose=True,
    backstory=(
        "With a strategic mind and an eye for detail, you "
        "excel at refining resumes to highlight the most "
        "relevant skills and experiences, ensuring they "
        "resonate perfectly with the job's requirements."
    )
)

In [28]:
# Agent 4: Interview Preparer
interview_preparer = Agent(
    role="Engineering Interview Preparer",
    goal="Create interview questions and talking points "
         "based on the resume and job requirements",
    tools = [scrape_tool, search_tool,
             read_resume, semantic_search_resume],
    verbose=True,
    backstory=(
        "Your role is crucial in anticipating the dynamics of "
        "interviews. With your ability to formulate key questions "
        "and talking points, you prepare candidates for success, "
        "ensuring they can confidently address all aspects of the "
        "job they are applying for."
    )
)

## Creating Tasks

In [29]:
# Task for Researcher Agent: Extract Job Requirements
research_task = Task(
    description=(
        "Analyze the job posting URL provided ({job_posting_url}) "
        "to extract key skills, experiences, and qualifications "
        "required. Use the tools to gather content and identify "
        "and categorize the requirements."
    ),
    expected_output=(
        "A structured list of job requirements, including necessary "
        "skills, qualifications, and experiences."
    ),
    agent=researcher,
    async_execution=True
)

In [30]:
# Task for Profiler Agent: Compile Comprehensive Profile
profile_task = Task(
    description=(
        "Compile a detailed personal and professional profile "
        "using the GitHub ({github_url}) URLs, and personal write-up "
        "({personal_writeup}). Utilize tools to extract and "
        "synthesize information from these sources."
    ),
    expected_output=(
        "A comprehensive profile document that includes skills, "
        "project experiences, contributions, interests, and "
        "communication style."
    ),
    agent=profiler,
    async_execution=True
)

In [31]:
# Task for Resume Strategist Agent: Align Resume with Job Requirements
resume_strategy_task = Task(
    description=(
        "Using the profile and job requirements obtained from "
        "previous tasks, tailor the resume to highlight the most "
        "relevant areas. Employ tools to adjust and enhance the "
        "resume content. Make sure this is the best resume even but "
        "don't make up any information. Update every section, "
        "inlcuding the initial summary, work experience, skills, "
        "and education. All to better reflrect the candidates "
        "abilities and how it matches the job posting."
    ),
    expected_output=(
        "An updated resume that effectively highlights the candidate's "
        "qualifications and experiences relevant to the job."
    ),
    output_file="tailored_resume.md",
    context=[research_task, profile_task],
    agent=resume_strategist
)

In [32]:
# Task for Interview Preparer Agent: Develop Interview Materials
interview_preparation_task = Task(
    description=(
        "Create a set of potential interview questions and talking "
        "points based on the tailored resume and job requirements. "
        "Utilize tools to generate relevant questions and discussion "
        "points. Make sure to use these question and talking points to "
        "help the candiadte highlight the main points of the resume "
        "and how it matches the job posting."
    ),
    expected_output=(
        "A document containing key questions and talking points "
        "that the candidate should prepare for the initial interview."
    ),
    output_file="interview_materials.md",
    context=[research_task, profile_task, resume_strategy_task],
    agent=interview_preparer
)


## Creating the Crew

In [33]:
job_application_crew = Crew(
    agents=[researcher,
            profiler,
            resume_strategist,
            interview_preparer],

    tasks=[research_task,
           profile_task,
           resume_strategy_task,
           interview_preparation_task],

    verbose=True
)

2025-05-15 11:53:09,201 - 8553434240 - __init__.py-__init__:535 - WARNING: Overriding of current TracerProvider is not allowed


## Running the Crew


In [34]:
job_application_inputs = {
    'job_posting_url': 'https://scoville.jp/careers/fullstack-web-engineer-ruby',
    'github_url': 'https://github.com/joaomdmoura',
    'personal_writeup': """Noah is an accomplished Software
    Engineering Leader with 18 years of experience, specializing in
    managing remote and in-office teams, and expert in multiple
    programming languages and frameworks. He holds an MBA and a strong
    background in AI and data science. Noah has successfully led
    major tech initiatives and startups, proving his ability to drive
    innovation and growth in the tech industry. Ideal for leadership
    roles that require a strategic and innovative approach."""
}

In [35]:
### this execution will take a few minutes to run
result = job_application_crew.kickoff(inputs=job_application_inputs)

 [DEBUG]: == Working Agent: Tech Job Researcher
 [INFO]: == Starting Task: Analyze the job posting URL provided (https://scoville.jp/careers/fullstack-web-engineer-ruby) to extract key skills, experiences, and qualifications required. Use the tools to gather content and identify and categorize the requirements.
 [DEBUG]: == [Tech Job Researcher] Task output: 


 [DEBUG]: == Working Agent: Personal Profiler for Engineers
 [INFO]: == Starting Task: Compile a detailed personal and professional profile using the GitHub (https://github.com/joaomdmoura) URLs, and personal write-up (Noah is an accomplished Software
    Engineering Leader with 18 years of experience, specializing in
    managing remote and in-office teams, and expert in multiple
    programming languages and frameworks. He holds an MBA and a strong
    background in AI and data science. Noah has successfully led
    major tech initiatives and startups, proving his ability to drive
    innovation and growth in the tech industry

In [36]:
from IPython.display import Markdown, display
display(Markdown("./tailored_resume.md"))

# Noah Williams
- Email: noah.williams@example.dev
- Phone: +44 11 111 11111

## Summary
Noah Williams is a distinguished Software Engineering Leader with an 18-year tenure in the technology industry. He excels in leading both remote and in-office engineering teams, with expertise in software development, process innovation, and enhancing team collaboration. Proficient in Ruby, Python, JavaScript, TypeScript, and Elixir, alongside deep expertise in various front end frameworks. Significant experience in data science and machine learning, spearheading successful deployments of scalable AI solutions and innovative data model development.

## Work Experience

### Director of Software Engineering
DataKernel (remote) — 2022 - Present
- Transformed engineering division into a key revenue pillar, expanding customer base and enhancing product capabilities with cutting-edge AI technologies.
- Led substantial growth in skill development, defining long-term strategic initiatives in adopting AI technologies.

### Senior Software Engineering Manager
DataKernel (remote) — 2019 - 2022
- Directed engineering strategy and operations, shaping company's technological trajectory.
- Managed diverse teams across multiple time zones, fostering performance and morale through talent recruitment and mentorship.

### Founder & CEO
InnovPet (remote) — 2019 - 2022
- Founded startup focused on IoT solutions for pet care, overseeing product development from concept to market entry.
- Set up advisory board, established production facilities overseas, navigated successful initial funding phase.

### Engineering Manager
EliteDevs (remote) — 2018 - 2019
- Formulated and executed strategic plans, enhancing inter-departmental coordination and trust.
- Managed multiple engineering teams, fostering innovation and productivity aligning with company's long-term goals.

### Engineering Manager
PrintPack (remote) — 2016 - 2018
- Led high-performance engineering team, increasing company revenue by 500% within two years.
- Integrated data analytics into business decision-making processes, leading to development of predictive modeling tool.

### Senior Software Engineer
DriveAI (remote) — 2015 - 2016
- Developed and optimized central API, improving functionality for large engineering team and users.
- Implemented critical enhancements, including advanced caching strategies for improved system performance.

### CTO
BetCraft — 2013 - 2015
- Led technology strategy post-Series A funding, guiding company through significant technological advancement and network expansion.
- Improved platform performance and expanded market reach through strategic initiatives and partnerships.

## Education

- MBA in Information Technology
  London Business School - MBA

- Advanced Leadership Techniques
  University of London - Certification

- Data Science Specialization
  Coursera (Johns Hopkins University) - Certification

- B.Sc. in Computer Science
  University of Edinburgh - Bachelor’s degree

Noah Williams is an ideal candidate for senior executive roles, particularly in companies seeking leadership with a robust blend of technical and strategic expertise.

In [37]:
display(Markdown("./interview_materials.md"))

Based on Noah Williams' tailored resume and the job requirements for a mid-career or senior Ruby on Rails engineer position, here are some potential interview questions and talking points:

1. Can you walk us through a project where you designed, built, and maintained efficient, reusable, and reliable Ruby code? How did you ensure the best possible performance and quality of the application?
2. Share an experience integrating data storage solutions and user-facing elements in a project. How did you approach troubleshooting and fixing bugs in this context?
3. How do you ensure that the code you write is clean, maintainable, and follows coding standards? Can you provide an example of a project where you demonstrated this?
4. Discuss your experience with JavaScript, HTML, and CSS. How have you utilized these technologies in conjunction with Ruby on Rails?
5. Explain your understanding of object-oriented programming principles and how you apply them in your projects. Give an example of a project where you implemented these concepts effectively.
6. How familiar are you with MVC, Mocking, ORM, and RESTful concepts? Can you provide examples of how you have implemented them in your projects?
7. Describe your proficiency with code versioning tools like Git. How do you ensure version control and collaboration within a team setting?
8. Have you worked with continuous integration tools before? If so, explain how you have implemented them to streamline development processes.
9. Share your experience with relational databases. How have you used them in your projects to store and retrieve data efficiently?
10. Have you had exposure to React? If yes, discuss a project where you integrated React with Ruby on Rails and the benefits it brought to the application.
11. How comfortable are you working in an Agile development environment? Share your experience with Agile methodologies and how you have contributed to Agile teams.
12. Can you discuss a time when you led a team through a challenging project or situation? How did you approach problem-solving and team collaboration to achieve success?
13. Share a project where you implemented AI technologies or innovative solutions. What was your role in the project, and what impact did it have on the final product?
14. How do you stay updated with the latest trends and technologies in the software engineering field? Can you provide examples of how you have applied new knowledge to your work?

These questions and talking points aim to assess Noah Williams' expertise, experience, and alignment with the job requirements, allowing the candidate to showcase their skills effectively during the interview process.